# This Notebook is used to create a visual representation of the Localization offset 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import PIL
import json
from PIL import Image
import os

In [ ]:
poses=[]
vertex_ids = []
vertex_times = []
times = []
folder_path = '/home/adam/Desktop/CurrentBranch/src/main/src/vtr_db_extractor/loc_results'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files: {file_count}")
for i in range(0,file_count,1):
    filename = filename=folder_path+f"/T_r_vertex_{i:04d}.txt"
    matrix = []
    
    with open(filename, 'r') as file:
        for line in file:
            # Skip empty lines
            if line.startswith('vertex_id:'):
                vertex_id =int(line.split(": ")[1].strip())
                vertex_ids.append(vertex_id)
            elif line.startswith('vertex_timestamp:'):
                vertex_time =int(line.split(": ")[1].strip())
                vertex_times.append(vertex_time)
            elif line.startswith("timestamp:"):
                time =int(line.split(": ")[1].strip())
                times.append(time)
            elif line.strip() and not line.startswith('u'):
                # Split the line by whitespace and convert each element to float
                row = [float(val) for val in line.strip().split()]
                matrix.append(row)
            

    temp = np.array(matrix[0:4])
    poses.append(np.array(matrix[0:4]))
poses=np.array(poses)
times= np.array(times)

vertex_ids = np.array(vertex_ids)
vertex_times= np.array(vertex_times)


time_c = []
start_time_c =0
first_frame= True
for time in times:
    if first_frame:
        start_time = time
        first_frame = False
        time_c.append(0.0)

    else:
        time_c.append((time-start_time)/1000000000)
time_c = np.array(time_c)

In [ ]:
# print(vertex_ids[4588:4600])
# print(vertex_ids[4585])
# print(vertex_ids[4586])
print(vertex_ids[332])
pose1 = poses.copy()
vertex_ids1 = vertex_ids.copy()

# pose1 = poses[0:4587].copy()
# pose2 = poses[4588:].copy()
# vertex_ids1 = vertex_ids[0:4587].copy()
# vertex_ids2 = vertex_ids[4588:].copy()
# print(vertex_ids2)
print(vertex_times)

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(8, 4), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
plt.hist(vertex_ids, bins=max(vertex_ids), color='b', edgecolor='black', alpha=0.6, label='Repeat 1') # color='skyblue'
# plt.hist(vertex_ids2, bins=max(vertex_ids2), color='salmon', edgecolor='salmon', alpha=0.6, label='Repeat 2')
plt.xlabel('Vertex Number')
plt.ylabel('Results')
plt.legend()
# plt.title('Vertex_ID frequency')
plt.show()

In [ ]:
%matplotlib ipympl
fig, axs = plt.subplots(1, 1, figsize =(8, 6), tight_layout = True)
axs.grid( color ='grey', linestyle ='-.', linewidth = 0.5, alpha = 0.6) 
plt.hist(vertex_ids1, bins=max(vertex_ids1), color='black', edgecolor='black', alpha=0.6, label='Repeat 1') # color='skyblue'
# plt.hist(vertex_ids2, bins=max(vertex_ids2), color='salmon', edgecolor='salmon', alpha=0.6, label='Repeat 2')
plt.xlabel('Vertex ID')
plt.ylabel('Results')
plt.legend()
plt.title('Vertex_ID frequency')
plt.show()

In [ ]:
origins = poses[:, :3, -1]
rep_1 = []
rep_t_1 = []
rep_2 = []
rep_t_2 = []
rep_3 = []
rep_t_3 = []
for i in range(len(origins[6:,1])):
    if time_c[6:][i]<5000:
        rep_1.append(origins[6:,1][i])
        rep_t_1.append(time_c[6:][i])
    elif time_c[6:][i]<10000:
        rep_2.append(origins[6:,1][i])
        rep_t_2.append(time_c[6:][i])
    else:
        rep_3.append(origins[6:,1][i])
        rep_t_3.append(time_c[6:][i])

print(len(rep_1))
print(len(rep_2))
print(len(rep_3))

%matplotlib ipympl
# Graph of the y distance to vertex.
origins = poses[:, :3, -1]
plt.plot(rep_t_3, rep_3, 'k--', linewidth=0.5)
plt.xlabel("time [seconds]")
plt.ylabel("Y-Offset")
plt.title("Deviation from path")
plt.show()

np.savetxt('plot_data.txt', np.column_stack([rep_t_3, rep_3]))

In [ ]:

%matplotlib ipympl
# Graph of the y distance to vertex.
origins = poses[:, :3, -1]
plt.plot(time_c[6:], origins[6:,1], 'k--', linewidth=0.5)
plt.xlabel("time [seconds]")
plt.ylabel("Y-Offset")
plt.title("Deviation from path")
plt.show()

# Write data to a text file

In [ ]:
np.savetxt('lateral_offset_data_old_canopy.txt', np.column_stack([time_c, origins[:,1]]))

In [ ]:

%matplotlib ipympl
# Graph of the y distance to vertex.
origins1 = pose1[:, :3, -1]
origins2 = pose2[:, :3, -1]
plt.plot(origins1[:,1], 'k--', linewidth=0.5)
plt.plot(origins2[:,1], 'k--', linewidth=0.5, color='salmon')
plt.xlabel("vertices")
plt.ylabel("Y-Offset")
plt.title("Deviation from path")
plt.show()

In [ ]:
%matplotlib ipympl
poses1=poses[0:100]
origins = poses1[:, :3, -1]
dirs = np.stack([np.sum([1, 0, 0] * pose[:3, :3], axis=-1) for pose in poses1])
print(origins)

ax = plt.figure(figsize=(12, 8)).add_subplot(projection='3d')
_ = ax.quiver(
  origins[..., 0].flatten(),
  origins[..., 1].flatten(),
  origins[..., 2].flatten(),
  dirs[..., 0].flatten(),
  dirs[..., 1].flatten(),
  dirs[..., 2].flatten(), length=0.0005, normalize=True)
ax.set_box_aspect([1,1,1])
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.view_init(45, 45) 
plt.show()